# Обследования рабочей силы

In [ ]:
import warnings
import inspect
import pandas as pd
import numpy as  np
from tqdm import tqdm

from dataclasses import dataclass
from collections import defaultdict

from statsmodels.api import OLS

from scipy.special import logit
from scipy.special import expit

import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_predict, LeaveOneOut, TimeSeriesSplit

warnings.filterwarnings("ignore")
plotly.offline.init_notebook_mode()

In [ ]:
columns = ["god", 
           "vesa_ob", 
           "struktak", 
           "nas_pol",
           "nas_vozr"]

data = []
folder = "микроданные-2023"
for year in tqdm(range(2010, 2024)):
    if year < 2022:
        d = pd.read_csv(f"{folder}/bd_ors_{year}.csv", 
                        usecols=columns)
    else:
        caps_columns = [column.upper() for column in columns]
        d = pd.read_csv(f"{folder}/bd_ors_{year}.csv",
                        usecols=caps_columns)
        d.rename(columns=dict(zip(caps_columns, columns)), inplace=True)
    data.append(d)

data = pd.concat(data, axis = 0)

In [ ]:
population_woman = pd.read_excel("population.xlsx", sheet_name="woman", index_col="age")
population_man   = pd.read_excel("population.xlsx", sheet_name="man",   index_col="age")

population_man.head()

In [ ]:
population_ors_man   = pd.DataFrame()
population_ors_woman = pd.DataFrame()

for year in range(2010, 2024):
    population_ors_man[year]   = data[(data["god"] == year) & (data["nas_pol"] == "Мужчины")].groupby("nas_vozr")["vesa_ob"].sum()
    population_ors_woman[year] = data[(data["god"] == year) & (data["nas_pol"] == "Женщины")].groupby("nas_vozr")["vesa_ob"].sum()

In [ ]:
year_intervals   = [[age, age+4] for age in range(15, 71, 5)]


fig = make_subplots(
    rows=2, cols=2,
    vertical_spacing=0.1,
    subplot_titles=["Мужчины, по годам",
                    "Женщины, по годам",
                    "Мужчины, по возрастным группам",
                    "Женщины, по возрастным группам"]
).update_layout(title="ОРС и ЕМИСС численность населения")

for i, year in enumerate(range(2010, 2024)):

    for col, (population, population_ors) in enumerate(zip([population_man, population_woman],
                                                           [population_ors_man, population_ors_woman]), 1):
        
        fig.add_trace(
            go.Bar( x = population_ors[year].index,
                    y = population_ors[year].loc[:74],
                    name = f"ОРС {year}",
                    legendgroup=f"ОРС {year}",
                    visible='legendonly',
                    marker ={"color" : px.colors.qualitative.Alphabet[i]},
                    showlegend = col == 1,
                    opacity=0.75),
            1, col)

        fig.add_trace(
            go.Bar( x = population[year].index,
                    y = population[year].loc[:74],
                    name = f"ЕМИСС {year}",
                    legendgroup=f"ЕМИСС {year}",
                    visible='legendonly',
                    marker={"color" : px.colors.qualitative.Alphabet[- (i * 2 + 1) % 26]},
                    showlegend= col == 1,
                    opacity=0.75),
            1, col)

        fig.add_trace(
            go.Bar( x = [f"{age_start}-{age_end}" for age_start, age_end in year_intervals],
                    y = [sum(population_ors[year].loc[age_start:age_end]) for age_start, age_end in year_intervals],
                    name = f"ОРС {year}",
                    legendgroup=f"ОРС {year}",
                    visible='legendonly',
                    marker ={"color" : px.colors.qualitative.Alphabet[i]},
                    showlegend=False,
                    opacity=0.75,),
            2, col)

        fig.add_trace(
            go.Bar( x = [f"{age_start}-{age_end}" for age_start, age_end in year_intervals],
                    y = [sum(population[year].loc[age_start:age_end]) for age_start, age_end in year_intervals],
                    name = f"ЕМИСС {year}",
                    legendgroup=f"ЕМИСС {year}",
                    visible='legendonly',
                    marker={"color" : px.colors.qualitative.Alphabet[- (i * 2 + 1) % 26]},
                    showlegend=False,
                    opacity=0.75,),
            2, col)
    
fig.update_layout(barmode='overlay')
fig.layout.update(width = 1500, height=750)
fig.show()

График отражает численность населения по возрастам за выбранный год по оценкам ОРС и официальных данных. В случае отдельных возрастов между источниками можно наблюдать расхождения, однако они уменьшаются, если агрегировать на возрастные группы.

In [ ]:
data["struktak"].replace({"Занятые" : "занятые",
                          "Лица, не входящие в состав рабочей силы" : "неактивные",
                          "экономически неактивные" : "неактивные",
                          "Экономически неактивные" : "неактивные",
                          "Безработные" : "безработные"}, inplace = True)
data["nas_vozr"] = data.loc[:, "nas_vozr"].astype(int)

In [ ]:
def func(x):
    y = x.groupby("struktak")["vesa_ob"].sum()
    y.index.name = None
    y.name = None
    return pd.DataFrame(y / y.sum()).T

fig = go.Figure().update_layout(title="Уровень участия в рабочей силе")

d = data
d = d.groupby("god").apply(func, include_groups=False).droplevel(1)
fig.add_trace(
    go.Scatter(y = (d["безработные"] + d["занятые"]) / d.sum(axis = 1), 
                x = d.index, 
                name = "Все"))

d = data[data["nas_pol"] == "Мужчины"]
d = d.groupby("god").apply(func, include_groups=False).droplevel(1)
fig.add_trace(
    go.Scatter(y = (d["безработные"] + d["занятые"]) / d.sum(axis = 1), 
                x = d.index, 
                name = "Мужчины"))

d = data[data["nas_pol"] == "Женщины"]
d = d.groupby("god").apply(func, include_groups=False).droplevel(1)
fig.add_trace(
    go.Scatter(y = (d["безработные"] + d["занятые"]) / d.sum(axis = 1), 
                x = d.index, 
                name = "Женщины"))

fig.show()

# Разбиение на половозрастные группы

In [ ]:
def calculate_labor(data):
    labor = data[(data["struktak"] == "занятые") | (data["struktak"] == "безработные")]["vesa_ob"].sum()
    no_labor = data[data["struktak"] == "неактивные"]["vesa_ob"].sum()
    return labor / (labor + no_labor)

fig = make_subplots(
        rows=1, cols=2,
        shared_xaxes=True,
        vertical_spacing=0.02,
        subplot_titles=['Мужчины', 'Женщины'],
        x_title = "возраст", 
        y_title = "доля в рабочей силе"
        ).update_layout(title="Участие в рабочей силе по возрастам")

for i, year in enumerate(range(2010, 2023)):
    d = data[data["god"] == year].groupby(["nas_vozr", "nas_pol"]).apply(calculate_labor, include_groups = False)
    d = d.loc[:75]
    fig.add_trace(go.Bar(x=d.loc[:, "Мужчины"].index, y=d.loc[:, "Мужчины"].values,
                name=f"{year}",
                visible='legendonly',
                legendgroup=f"{year}",
                marker ={"color" : px.colors.qualitative.Alphabet[i]},
                opacity=0.75),
                row = 1, col=1)

    fig.add_trace(go.Bar(x=d.loc[:, "Женщины"].index, y=d.loc[:, "Женщины"].values,
                name=f"{year}",
                visible='legendonly',
                legendgroup=f"{year}",
                marker ={"color" : px.colors.qualitative.Alphabet[i]},
                showlegend=False,
                opacity=0.75),
                row = 1, col=2)
    
fig.update_layout(barmode='overlay')
fig.update_layout()
fig.show();

График отражает участие в рабочей силе населения разного возраста и позволяет сравнить несколько лет между собой.

Разобьем на половозрастные группы

In [ ]:
man_year_intervals = [[age, age+4] for age in range(15, 71, 5)]
man_year_intervals[-1] = [70, 75]

woman_year_intervals = [[age, age+4] for age in range(15, 71, 5)]
woman_year_intervals[-1] = [70, 75]

def calculate_labor(data):
    labor = data[(data["struktak"] == "занятые") | (data["struktak"] == "безработные")]["vesa_ob"].sum()
    no_labor = data[data["struktak"] == "неактивные"]["vesa_ob"].sum()
    return labor / (labor + no_labor)    


man_data = pd.DataFrame()
for year in tqdm(range(2010, 2024)):
    d = data[(data["nas_pol"] == "Мужчины") & (data["god"] == year)]
    for interval in man_year_intervals:
        man_data.loc[year, f"{interval[0]}-{interval[1]}"] = calculate_labor(d[(d["nas_vozr"] >= interval[0]) &
                                                                               (d["nas_vozr"] <= interval[1])])
        
woman_data = pd.DataFrame()
for year in tqdm(range(2010, 2024)):
    d = data[(data["nas_pol"] == "Женщины") & (data["god"] == year)]
    for interval in man_year_intervals:
        woman_data.loc[year, f"{interval[0]}-{interval[1]}"] = calculate_labor(d[(d["nas_vozr"] >= interval[0]) &
                                                                                 (d["nas_vozr"] <= interval[1])])

Построим график участия половозрастных групп

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=['Участие в рабочей силе (мужчины)', 
                    'Участие в рабочей силе (женщины)']
)

for i, age_group in enumerate(man_data.columns, start = 4):
    fig.add_trace(
        go.Scatter(y = man_data[age_group], 
                   x = man_data.index, 
                   name = age_group,
                   line=dict(color=px.colors.qualitative.Alphabet[i % 26]),
                   legendgroup=f'group{age_group}',
                   visible='legendonly'),
                   1, 1)
    
for i, age_group in enumerate(woman_data.columns, start = 4):
    fig.add_trace(
        go.Scatter(y = woman_data[age_group], 
                   x = woman_data.index, 
                   name = age_group,
                   line=dict(color=px.colors.qualitative.Alphabet[i % 26]),
                   legendgroup=f'group{age_group}',
                   visible='legendonly',
                   showlegend=False),
                   2, 1)

fig.layout.update(width = 1200, height=750)

fig.show()

График отражает тенденции участия в рабочей силе разных половозрастных групп.

# Экономические факторы

Макроэкономические факторы

1) `gdp` - ВВП

2) `deflator_gdp` - дефлятор ВВП

3) `dollar` - курс доллара

4) `cpi` - ИПЦ общий

5) `cpi_food` - ИПЦ на продовольственные товары

6) `cpi_good` - ИПЦ на непродовольственные товары

7) `cpi_service` - ИПЦ на услуги

8) `real_income` - реальный доход

9) `unemployment_level` - уровень безработицы

10) `disposable_income` - располагаемый доход

11) `labor_effic` - производительность труда

12) `retirement_living_wage` - прожиточный минимум пенсионеров

13) `labor_living_wage` - прожиточный минимум трудоспособного населения


In [ ]:
economic_factors = pd.read_excel("factors/econ_factors.xlsx", header=None)
econ_column_name = {column : name for name, column in zip(economic_factors.loc[0], economic_factors.loc[1])}
economic_factors.columns = economic_factors.loc[1]
economic_factors.drop([0, 1], axis = 0, inplace = True)
economic_factors.set_index("year", inplace = True, drop=True)
economic_factors = economic_factors.astype(float)
economic_factors.head()

In [ ]:
# fig = make_subplots(
#     rows=economic_factors.shape[1] // 3 + 1, cols=3,
#     shared_xaxes=True,
#     vertical_spacing=0.02,
#     subplot_titles=[econ_column_name[col] for col in economic_factors.columns] 
# )

# for i, column in enumerate(economic_factors.columns):
#     fig.add_trace(
#         go.Scatter(y = economic_factors[column], 
#                    x = economic_factors.index, 
#                    name = column,
#                    line=dict(color=px.colors.qualitative.Alphabet[i % 26]),
#                    legendgroup=column,
#                    showlegend=False),
#                    i // 3 + 1, i % 3 + 1)

# fig.layout.update(width = 1500, height=1500)

# fig.show()

# Демографические факторы

Демографические факторы

1) `younger_old` - доля людей старше трудоспособного возраста

2) `in_old` - доля людей в трудоспособном возрасте

3) `older_old` - доля людей старше трудоспособного возраста

4) `coeff_born` - коэффициент рождаемости (на 1000 чел)

5) `coeff_death` - коэффициент смертности (на 1000 чел)

6) `man_life_exp` - ожидаемая продолжительность жизни (мужчины)

7) `woman_life_exp` - ожидаемая продолжительность жизни (женщины)

8) `coeff_child` - количество детей на 1 женщину

9) `coeff_migration` - коэффициент мигрантов (на 1000 человек)

10) `retirement_age_man` - пенсионный возраст (мужчины)

11) `retirement_age_woman` - пенсионный возраст (женщины)

In [ ]:
demographic_factors = pd.read_excel(f"factors/dem_factors.xlsx", header=None)
dem_column_name = {column : name for name, column in zip(demographic_factors.loc[0], demographic_factors.loc[1])}
demographic_factors.columns = demographic_factors.loc[1]
demographic_factors.drop([0, 1], axis = 0, inplace = True)
demographic_factors.set_index("year", inplace = True, drop=True)
demographic_factors = demographic_factors.astype(float)
demographic_factors.head()

In [ ]:
# fig = make_subplots(
#     rows=demographic_factors.shape[1] // 3 + 1, cols=3,
#     shared_xaxes=True,
#     vertical_spacing=0.02,
#     subplot_titles=[dem_column_name[col] for col in demographic_factors.columns] 
# )

# for i, column in enumerate(demographic_factors.columns):
#     fig.add_trace(
#         go.Scatter(y = demographic_factors[column], 
#                    x = demographic_factors.index, 
#                    name = column,   
#                    line=dict(color=px.colors.qualitative.Alphabet[i % 26]),
#                    legendgroup=column,
#                    showlegend=False),
#                    i // 3 + 1, i % 3 + 1)

# fig.layout.update(width = 1200, height=1500)

# fig.show()

In [ ]:
woman_death_coeffs = pd.read_excel(f"factors/dem_factors.xlsx", skiprows=1, index_col="year", sheet_name = "woman_death_coeffs")
woman_death_coeffs.head()

man_death_coeffs = pd.read_excel(f"factors/dem_factors.xlsx", skiprows=1, index_col="year", sheet_name = "man_death_coeffs")
man_death_coeffs.head()

woman_child_coeffs = pd.read_excel(f"factors/dem_factors.xlsx", skiprows=1, index_col="year", sheet_name = "woman_child") / 1000
woman_child_coeffs.head()

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=["Возрастные коэффициенты смертности (мужчины)", 
                    "Возрастные коэффициенты смертности (женщины)"]
)

for i, column in enumerate(man_death_coeffs.columns):
    fig.add_trace(
        go.Scatter( y = man_death_coeffs[column], 
                    x = man_death_coeffs.index, 
                    name = column,
                    visible='legendonly',
                    line=dict(color=px.colors.qualitative.Alphabet[i % 26]),
                    legendgroup=column),
                    1, 1)

for i, column in enumerate(man_death_coeffs.columns):
    fig.add_trace(
        go.Scatter( y = woman_death_coeffs[column], 
                    x = woman_death_coeffs.index, 
                    name = column,
                    visible='legendonly',
                    line=dict(color=px.colors.qualitative.Alphabet[i % 26]),
                    legendgroup=column,
                    showlegend=False),
                    2, 1)

fig.layout.update(width = 1200, height=750)

fig.show()

In [ ]:
fig = make_subplots(
    rows=3, cols=1,
    vertical_spacing=0.1,
    subplot_titles=["Сруктура родившихся живыми на 1 женщину по возрастам (2023)",
                    "Родившиеся живыми на 1 женщину по годам",
                    "Пенсионный возраст"]
)
for year in range(2010, 2024, 2):
    fig.add_trace(
        go.Bar( x = woman_child_coeffs.columns,
                y = woman_child_coeffs.loc[year] / sum(woman_child_coeffs.loc[year]),
                name = f"Количество родившихся ({year})",
                opacity=0.75,),
        1, 1)

fig.add_trace(
    go.Scatter( y = demographic_factors["coeff_child"], 
                x = demographic_factors.index, 
                name = "Количество родившихся",
                showlegend = False),
    2, 1)

fig.add_trace(
    go.Scatter( y = demographic_factors["retirement_age_man"], 
                x = demographic_factors.index, 
                name = "мужчины",
                showlegend = False),
    3, 1)

fig.add_trace(
    go.Scatter( y = demographic_factors["retirement_age_woman"], 
                x = demographic_factors.index, 
                name = "женщины",
                showlegend = False),
    3, 1)

fig.layout.update(width = 1200, height=750)
fig.update_layout(
    barmode='overlay')
fig.show()

In [ ]:
@dataclass
class Dataset:
    features : pd.DataFrame
    target   : pd.Series

    def __len__(self):
        return self.features.shape[0]

def calcualte_grade_val(data_age_population):
    """
    Функция вычисляет grade переменную по возрастам внутри группы
    """
    def phi(p):
        res = p * np.log(p) + (1-p) * np.log(1-p)
        res[p == 0] = 0
        res[p == 1] = 0
        return res

    grade_values = pd.DataFrame(columns=["grade_age"])

    for year in data_age_population.columns:
        age_group = data_age_population[year].values
        P = age_group.cumsum() / age_group.sum()
        p = P.copy()
        p[1:] = p[:-1]
        p[0] = 0
        grade_values.loc[year] = np.nansum((phi(P) - phi(p)) / (P - p), )
    return grade_values

Признаки, которые участвуют в обучении:

* `unemployment_level` - уровень безработицы

* `retirement_age` - пенсионный возраст

* `death_coeffs` - коэффициенты смертности (для групп 55+)

* `coeff_child` - количество детей у одной женщины (для групп 20-39)

* `grade_age` - грейд переменная, которая строится по возрасту внутри группы

* `grade_retirement` - грейд переменная, которая строится разнице между возрастом внутри группы и возрастом выхода на пенсию.

In [ ]:
def calcualte_grade_val_retirement(population, retirement_ages):

    df_age_group = pd.DataFrame(index = range(-10, 10))
    population = population.copy()

    for year in population.columns:
        
        retirement_age = int(retirement_ages[year])
        df = population[year]
        df.index = retirement_age - df.index
        df_age_group[year] = df

    grade_val_retirement = calcualte_grade_val(df_age_group.fillna(0))
    grade_val_retirement.columns = ["grade_retirement"]
    return grade_val_retirement

Сбор датасетов для мужчин

In [ ]:
man_datasets = dict()

for age_group in man_data.columns:
    
    age_start, age_end = [int(age) for age in age_group.split('-')]
    features = [economic_factors[["unemployment_level"]],
                calcualte_grade_val(population_man.loc[age_start:age_end])]
    
    if  age_start >= 50:
        death_coeff = man_death_coeffs[[age_group]]
        death_coeff.columns = ["death_coeff"]
        features.append(death_coeff)
    
    if age_start >= 55 and age_end <= 69:
        features.append(calcualte_grade_val_retirement(population_man.loc[age_start:age_end], demographic_factors["retirement_age_man"]))
        
    if age_start >= 55:
        retirement_age = demographic_factors[["retirement_age_man"]]
        retirement_age.columns = ["retirement_age"]
        features.append(retirement_age)
    
    man_datasets[age_group] = Dataset(features = pd.concat(features, axis = 1).loc[:2023],
                                      target   = man_data[age_group])

Сбор датасетов для женщин

In [ ]:
woman_datasets = dict()
for age_group in woman_data.columns:
    
    age_start, age_end = [int(age) for age in age_group.split('-')]
    features = [economic_factors[["unemployment_level"]],
                calcualte_grade_val(population_woman.loc[age_start:age_end])]
    
    if age_start >= 20 and age_end <= 39:
        child_coeff = woman_child_coeffs[[age_group]]
        child_coeff.columns = ["child_coeff"]
        features.append(child_coeff)
    
    if age_start >= 50 and age_end <= 64:
        features.append(calcualte_grade_val_retirement(population_woman.loc[age_start:age_end], demographic_factors["retirement_age_woman"]))

    if  age_start >= 50:
        death_coeff = woman_death_coeffs[[age_group]]
        death_coeff.columns = ["death_coeff"]
        features.append(death_coeff)

        retirement_age = demographic_factors[["retirement_age_woman"]]
        retirement_age.columns = ["retirement_age"]
        features.append(retirement_age)
        
    woman_datasets[age_group] = Dataset(features = pd.concat(features, axis = 1).loc[:2023],
                                        target   = woman_data[age_group])

In [ ]:
class Transformer:
    def __init__(self):
        self.feature_transform = {}
    

    def add_features(self, dataset : Dataset):
        for name, feature in dataset.features.items():
            max_corr = np.abs(pd.Series.corr(feature, logit(dataset.target)))
            self.feature_transform[name] = lambda x, p = 1: np.pow(x, p)
            
            if (feature < 0).any():
                continue
            
            max_corr = np.abs(pd.Series.corr(np.log(feature), logit(dataset.target)))
            self.feature_transform[name] = np.log

            for p in np.linspace(-5, 5, 2000):
                curr_corr = np.abs(pd.Series.corr(np.pow(feature, p), logit(dataset.target)))

                if curr_corr > max_corr:
                    self.feature_transform[name] = lambda x, p = p: np.pow(x, p)
                    max_corr = curr_corr
        return self
    

    def transform(self, X, y = None):
        X = X.copy()

        if y is not None:
            y = y.copy()

        for feature, transform in self.feature_transform.items():
            X[feature] = transform(X[feature])
        
        if y is not None:
            y = logit(y)
        
        return X, y
    

    def __call__(self, X, y = None):
        return self.transform(X, y)
    

    def calc_derivative(self, X):
        X = X.copy()
        
        for feature, transform in self.feature_transform.items():
            if transform == np.log:
                X[feature] = 1/X[feature]
            else:
                p = inspect.signature(transform).parameters['p'].default
                X[feature] = p * np.pow(X[feature], (p - 1))
        return X


def create_transformers(datasets):
    transforms = dict()
    for age_group, dataset in datasets.items():
        transformer = Transformer()
        transformer.add_features(dataset)
        transforms[age_group] = transformer
    return transforms


man_transforms   = create_transformers(man_datasets)
woman_transforms = create_transformers(woman_datasets)

In [ ]:
def create_corr_table(datasets, transforms):
    corrs = pd.DataFrame(index = datasets.keys())

    for age_group in datasets.keys():
        dataset     = datasets[age_group]
        transformer = transforms[age_group]
        features_transf, target_transf = transformer(dataset.features, dataset.target)
        
        for feature_name, feature_transf in features_transf.items():
            corrs.loc[age_group, feature_name] = pd.Series.corr(feature_transf, target_transf)
        
    return corrs

man_corrs   = create_corr_table(man_datasets,   man_transforms)
woman_corrs = create_corr_table(woman_datasets, woman_transforms)

fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        x_title = "возраст", 
        y_title = "доля в рабочей силе",
        subplot_titles = ["Корреляция с уровнями безработицы",
                          "Корреляция с числом детей",
                          "Корреляция с коэффициентами смертности",
                          "Корреляция с возрастом выхода на пенсию",
                          "Корреляция с grade переменной",
                          "Корреляция с grade retirement переменой"]
        ).update_layout(title="Корреляция logit(участия в рабочей силе) и np.log(экономических факторов)")

feature_row = { "unemployment_level"   : 1,
                "child_coeff"          : 2,
                "death_coeff"          : 3,
                "retirement_age"       : 4,
                "grade_age"            : 5,
                "grade_retirement" : 6}

for feature in man_corrs.columns:
    fig.add_trace(  go.Bar( x=man_corrs.index, 
                            y=man_corrs[feature],
                            name="мужчины",
                            marker={"color" : px.colors.qualitative.Alphabet[0]},
                            showlegend = False),
                    row = feature_row[feature], 
                    col=1)
    
for feature in woman_corrs.columns:
    fig.add_trace(  go.Bar( x=woman_corrs.index, 
                            y=woman_corrs[feature],
                            name="женщины",
                            marker={"color" : px.colors.qualitative.Alphabet[1]},
                            showlegend = False),
                    row = feature_row[feature], 
                    col=1)
    
fig.layout.update(width = 800, height=1000)
fig.show()

# Построение линейной регрессии

In [ ]:
from sklearn.base import BaseEstimator
from statsmodels.tools.tools import add_constant

class StatsModelsEstimator(BaseEstimator):

    def __init__(self, model_class, **init_params):
        self.model_class = model_class
        self.init_params = init_params

    def fit(self, X, y):
        X = X.copy()

        self.model_ = self.model_class(endog = y, exog = add_constant(X, has_constant='add'), **self.init_params).fit()
        return self

    def predict(self, X):
        X = X.copy()
        return self.model_.predict(exog = add_constant(X, has_constant='add'))

    def summary2(self):
        return self.model_.summary2()

In [ ]:
def train_models(datasets, transforms, fig):
    regressions = dict()
    model_metrics = defaultdict(lambda : defaultdict(lambda : dict()))
    naive_metrics = defaultdict(lambda : defaultdict(lambda : dict()))
    
    for i, age_group in enumerate(datasets.keys()):
        t = 5
        features, target = datasets[age_group].features, datasets[age_group].target
        X, y = transforms[age_group](features, target)

        regressions[age_group] = StatsModelsEstimator(OLS).fit(X, y)

        fit_predicts = expit( regressions[age_group].predict(X))
        cross_val_predicts = expit(cross_val_predict(StatsModelsEstimator(OLS), X, y, cv = LeaveOneOut()))
        time_val_predicts  = expit([StatsModelsEstimator(OLS).fit(X.iloc[train_indexes], y.iloc[train_indexes]).predict(X.iloc[test_indexes])
                                    for train_indexes, test_indexes in TimeSeriesSplit(n_splits=y.shape[0] - 1, test_size=1).split(X, y)]).reshape(-1)[t-1:]
        naive_predicts = [target.values[i-1] for i in range(1, len(y))]
        
        naive_metrics["mae"]["кросс валидация naive"][age_group] = mean_absolute_error(target[1:], naive_predicts)
        model_metrics["mae"]["кросс валидация model"][age_group] = mean_absolute_error(target, cross_val_predicts)

        model_metrics["mae"]["фиттинг model"][age_group] = mean_absolute_error(target, fit_predicts)
        model_metrics["mae"]["временная валидация model"][age_group]  = mean_absolute_error(target[t:], time_val_predicts)
        
        
        if fig is not None:

            fig.add_trace(
                go.Scatter( y = fit_predicts, 
                            x = X.index, 
                            name = f"{age_group} model",
                            visible='legendonly',
                            line=dict(color=px.colors.qualitative.Alphabet[1], dash="dot"),
                            legendgroup=f"{age_group}",
                            showlegend = i == 0),
                            1, 1)
        

            fig.add_trace(
                go.Scatter( y = cross_val_predicts, 
                            x = X.index, 
                            name = f"{age_group} model",
                            visible='legendonly',
                            line=dict(color=px.colors.qualitative.Alphabet[1], dash="dot"),
                            legendgroup=f"{age_group}",
                            showlegend = False),
                            2, 1)
        
            fig.add_trace(
                go.Scatter( y = time_val_predicts, 
                            x = X.index[t:], 
                            name = f"{age_group} model",
                            visible='legendonly',
                            line=dict(color=px.colors.qualitative.Alphabet[1], dash="dot"),
                            legendgroup=f"{age_group}",
                            showlegend = False),
                            3, 1)

            for j in range(1, 4):
                fig.add_trace(
                    go.Scatter( y = naive_predicts, 
                                x = X.index[1:], 
                                name = f"{age_group} naive",
                                visible='legendonly',
                                line=dict(color=px.colors.qualitative.Alphabet[2], dash="dot"),
                                legendgroup=f"{age_group}",
                                showlegend = i == 0 and j == 1),
                                j, 1)
                fig.add_trace(
                    go.Scatter( y = target, 
                                x = X.index, 
                                name = f"{age_group} real",
                                visible='legendonly',
                                line=dict(color=px.colors.qualitative.Alphabet[0]),
                                legendgroup=f"{age_group}",
                                showlegend= j == 1),
                                j, 1)
                
    return regressions, model_metrics, naive_metrics

In [ ]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=["fitting",
                    "Leave one out",
                    "Time predict"]
)

man_regressions, man_model_metrics, man_naive_metrics = train_models(man_datasets, man_transforms, fig)
fig.layout.update(width = 1200, height=800)
fig.show();

In [ ]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=["fitting",
                    "Leave one out",
                    "Time predict"]
)

woman_regressions, woman_model_metrics, woman_naive_metrics = train_models(woman_datasets, woman_transforms, fig)
fig.layout.update(width = 1200, height=800)
fig.show();

Вычисление коэффициента элластичности:
$$
logit(y) = ... + \beta_l x_{l}^{l'} + \beta_k x_k^{k'} + ...\\
y = logit^{-1}(... + \beta_l x_{l}^{l'} + \beta_k x_k^{k'} + ...) = expit(... + \beta_l x_{l}^{l'} + \beta_k x_k^{k'} + ...)\\

ELcoeff(\overline{x_k}) = \frac{\beta_k k' \overline{x_k}^{k'} expit(... + \beta_l \overline{x_{l}}^{l'} + \beta_k \overline{x_{k}}^{k'} + ...) \cdot (1 - expit(... + \beta_l \overline{x_{l}}^{l'} + \beta_k \overline{x_{k}}^{k'} + ...))}{expit(... + \beta_l \overline{x_{l}}^{l'} + \beta_k \overline{x_{k}}^{k'} + ...)} = \beta_k k' \overline{x_k}^{k'} \cdot (1 - expit(... + \beta_l \overline{x_{l}}^{l'} + \beta_k \overline{x_{k}}^{k'} + ...))
$$


In [ ]:
#проверка формулы коэффициента элластичности
def calc_elastic_coeff_numerically(factor, age_group, regressions, datasets, transforms):
    regression = regressions[age_group]
    dataset    = datasets   [age_group]
    transform  = transforms [age_group]

    X_mean = dataset.features.mean(axis=0).to_frame().T

    X_mean_transf = transform(X_mean)[0]
    res = expit(float(regression.predict(X_mean_transf)))

    X_mean_1 = X_mean.copy()
    X_mean_1[factor] *= 1.01

    X_mean_transf_1 = transform(X_mean_1)[0]
    res_1 = expit(float(regression.predict(X_mean_transf_1)))

    return (res_1 / res - 1) * 100

In [ ]:
fig = make_subplots(
    rows=3, cols=2,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.1,
    
    subplot_titles=["Коэффициенты для мужчин",               "Коэффициенты для женщин", 
                    "Коэффициенты элластичности для мужчин", "Коэффициенты элластичности для женщин",
                    "Mexval для мужчин",                     "Mexval для женщин"]
).update_layout(title="Коэффициенты регрессии")


def calc_mexval(t_statistic, T, n):
    return 100 * (np.sqrt(1 + t_statistic**2/(n - T)) - 1)


def plot_graphic(regressions, transforms, datasets, col_i, fig, debug=False):
    coeffs  = pd.DataFrame(index = regressions.keys())
    mexvals = pd.DataFrame(index = regressions.keys())
    elasts  = pd.DataFrame(index = regressions.keys())
    
    if debug:
        elasts_num = pd.DataFrame(index = regressions.keys())

    for age_group in regressions.keys():
        regression = regressions[age_group]
        dataset    = datasets[age_group]
        transform  = transforms[age_group]

        table = regression.summary2().tables[1]
        for feature in table.index:
            if feature == "const":
                continue
                
            X_mean = dataset.features.mean(axis=0).to_frame().T
            derivative = transform.calc_derivative(X_mean).loc[0, feature]
            pred = regression.predict(transform(X_mean)[0])[0]

            coeffs .loc[age_group, feature] = table.loc[feature, "Coef."]
            mexvals.loc[age_group, feature] = calc_mexval(table.loc[feature, "t"], table.shape[0], len(dataset.features))
            elasts .loc[age_group, feature] = table.loc[feature, "Coef."] * derivative * (1 - expit(pred)) * X_mean.loc[0, feature]
            
            if debug:
                elasts_num.loc[age_group, feature] = calc_elastic_coeff_numerically(feature, age_group, regressions, datasets, transforms)

    for feature in coeffs.columns:
        showlegend = feature not in plot_graphic.label_color.keys()
        fig.add_trace(
            go.Bar(     y = coeffs[feature], 
                        x = coeffs.index, 
                        name = f"{feature}",
                        visible='legendonly',
                        marker={"color" : px.colors.qualitative.Alphabet[plot_graphic.label_color[feature]]},
                        legendgroup=f"{feature}",
                        showlegend=showlegend),
                        1, col_i)
        
    
        fig.add_trace(
            go.Bar(     y = elasts[feature], 
                        x = elasts.index, 
                        name = f"{feature}",
                        visible='legendonly',
                        marker={"color" : px.colors.qualitative.Alphabet[plot_graphic.label_color[feature]]},
                        legendgroup=f"{feature}",
                        showlegend=False),
                        2, col_i)
        
        if debug:
            fig.add_trace(
            go.Bar(     y = elasts_num[feature], 
                        x = elasts_num.index, 
                        name = f"debug",
                        visible='legendonly',
                        marker={"color" : "red"},
                        legendgroup=f"{feature}",
                        showlegend=False),
                        2, col_i)
        fig.add_trace(
            go.Bar(     y = mexvals[feature], 
                        x = coeffs.index, 
                        name = f"{feature}",
                        visible='legendonly',
                        marker={"color" : px.colors.qualitative.Alphabet[plot_graphic.label_color[feature]]},
                        legendgroup=f"{feature}",
                        showlegend=False),
                        3, col_i)

plot_graphic.label_color = defaultdict(lambda : len(plot_graphic.label_color) + 1)
plot_graphic(man_regressions,   man_transforms,   man_datasets,   1, fig)
plot_graphic(woman_regressions, woman_transforms, woman_datasets, 2, fig)

fig.layout.update(width = 1400, height=800)
fig.show();

Коэффициент элластичности- насколько процентов изменится y, если x меняется на 1%

Mexval - насколько изменится MSE в %, если убрать x

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.1,
    
    subplot_titles=["Метрики для мужчин", "Метрики для женщин", 
                    "MAE", "MAE",
                    "r2", "r2"]
).update_layout(title="Метрики качества")

def plot_durbin_watson(regressions, fig, showlegend, row, col, color = 0):
    age_groups = sorted(list(regressions.keys()))
    for indexes, name in zip([[0, 3]], 
                            ["Durbin-Watson"]):
        

        coeffs = [float(regressions[age_group].summary2().tables[2].loc[*indexes]) for age_group in age_groups]

        fig.add_trace(
            go.Bar(     y = coeffs, 
                        x = age_groups, 
                        name = name,
                        visible='legendonly',
                        marker={"color" : px.colors.qualitative.Alphabet[color % 26]},
                        legendgroup=name,
                        text = np.round(coeffs, 3),
                        showlegend=showlegend),
                        row, col)
        color += 1

plot_durbin_watson(man_regressions, fig, True, 1, 1, 1)
plot_durbin_watson(woman_regressions, fig, False, 1, 2, 1)

def plot_metrics(metrics, fig, showlegend, row, col, color = 0):
    age_groups = sorted(list(list(metrics.values())[0].keys()))
    for metric in metrics.keys():
        fig.add_trace(
            go.Bar(     y = [metrics[metric][age_group] for age_group in age_groups], 
                        x = age_groups, 
                        name = f"{metric}",
                        visible='legendonly',
                        marker={"color" : px.colors.qualitative.Alphabet[color % 26]},
                        legendgroup=metric,
                        text=np.round([metrics[metric][age_group] for age_group in age_groups], 4),
                        showlegend=showlegend),
                        row, col)
        color += 1

plot_metrics(man_model_metrics["mae"],     fig, True,  2, 1, 3)
plot_metrics(woman_model_metrics["mae"],   fig, False, 2, 2, 3)
plot_metrics(man_naive_metrics["mae"],     fig, True,  2, 1, 10)
plot_metrics(woman_naive_metrics["mae"],   fig, False,  2, 2, 10)

fig.layout.update(width = 1500, height=600)
fig.show()

# Построение сценарных прогнозов

The craft of economic modeling